# Dither Algorithm Explorer

This notebook compares ordered dithering, several classic error-diffusion methods, and a simple blue-noise style approach. The goal is to make the pattern structure of each algorithm easy to see on the same grayscale gradient.

In [ ]:
import sys
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

sys.path.insert(0, '../tools')

try:
    import dither_applier as da
    print('Imported dither_applier from ../tools')
except Exception as exc:
    da = None
    print(f'Using inline fallback functions: {exc}')

def to_gray_array(image):
    if isinstance(image, Image.Image):
        return np.asarray(image.convert('L'), dtype=np.float32) / 255.0
    arr = np.asarray(image, dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[..., :3].mean(axis=2)
    return np.clip(arr, 0.0, 255.0) / 255.0 if arr.max() > 1.0 else np.clip(arr, 0.0, 1.0)

def bayer_matrix(size):
    if size == 2:
        return np.array([[0, 2], [3, 1]], dtype=np.float32)
    smaller = bayer_matrix(size // 2)
    return np.block([
        [4 * smaller + 0, 4 * smaller + 2],
        [4 * smaller + 3, 4 * smaller + 1],
    ])

def ordered_dither(image, matrix_size=4, levels=2):
    arr = to_gray_array(image)
    mat = bayer_matrix(matrix_size)
    threshold = (mat + 0.5) / (matrix_size * matrix_size)
    tiled = np.tile(threshold, (arr.shape[0] // matrix_size + 1, arr.shape[1] // matrix_size + 1))
    tiled = tiled[:arr.shape[0], :arr.shape[1]]
    if levels <= 1:
        return (arr > tiled).astype(np.float32)
    scaled = arr * (levels - 1)
    base = np.floor(scaled)
    frac = scaled - base
    return np.clip(base + (frac > tiled), 0, levels - 1) / (levels - 1)

def error_diffusion(image, kernel, levels=2):
    work = to_gray_array(image).copy()
    h, w = work.shape
    denom = max(levels - 1, 1)
    for y in range(h):
        for x in range(w):
            old = work[y, x]
            new = np.round(old * denom) / denom
            err = old - new
            work[y, x] = new
            for dy, dx, weight in kernel:
                ny, nx = y + dy, x + dx
                if 0 <= ny < h and 0 <= nx < w:
                    work[ny, nx] = np.clip(work[ny, nx] + err * weight, 0.0, 1.0)
    return work

FS_KERNEL = [(0, 1, 7 / 16), (1, -1, 3 / 16), (1, 0, 5 / 16), (1, 1, 1 / 16)]
ATKINSON_KERNEL = [(0, 1, 1 / 8), (0, 2, 1 / 8), (1, -1, 1 / 8), (1, 0, 1 / 8), (1, 1, 1 / 8), (2, 0, 1 / 8)]
JJN_KERNEL = [
    (0, 1, 7 / 48), (0, 2, 5 / 48),
    (1, -2, 3 / 48), (1, -1, 5 / 48), (1, 0, 7 / 48), (1, 1, 5 / 48), (1, 2, 3 / 48),
    (2, -2, 1 / 48), (2, -1, 3 / 48), (2, 0, 5 / 48), (2, 1, 3 / 48), (2, 2, 1 / 48),
]
SIERRA_KERNEL = [
    (0, 1, 5 / 32), (0, 2, 3 / 32),
    (1, -2, 2 / 32), (1, -1, 4 / 32), (1, 0, 5 / 32), (1, 1, 4 / 32), (1, 2, 2 / 32),
    (2, -1, 2 / 32), (2, 0, 3 / 32), (2, 1, 2 / 32),
]

def floyd_steinberg(image, levels=2):
    return error_diffusion(image, FS_KERNEL, levels=levels)

def atkinson(image, levels=2):
    return error_diffusion(image, ATKINSON_KERNEL, levels=levels)

def jjn(image, levels=2):
    return error_diffusion(image, JJN_KERNEL, levels=levels)

def sierra(image, levels=2):
    return error_diffusion(image, SIERRA_KERNEL, levels=levels)

def ign_blue_noise(h, w):
    yy, xx = np.indices((h, w))
    return np.mod(52.9829189 * np.mod(0.06711056 * xx + 0.00583715 * yy, 1.0), 1.0)

def blue_noise_dither(image, levels=2):
    arr = to_gray_array(image)
    noise = ign_blue_noise(*arr.shape)
    denom = max(levels - 1, 1)
    if levels <= 1:
        return (arr > noise).astype(np.float32)
    scaled = arr * denom
    base = np.floor(scaled)
    frac = scaled - base
    return np.clip(base + (frac > noise), 0, denom) / denom

plt.rcParams['figure.figsize'] = (10, 4)

## The Test Gradient

A good dithering test spans the full tonal range and does so slowly enough that pattern changes are easy to inspect. A long horizontal gradient reveals where each algorithm produces banding, worms, texture, or pleasing blue-noise-like dispersion.

In [ ]:
grad = np.tile(np.linspace(0, 255, 512, dtype=np.uint8), (64, 1))
gradient_image = Image.fromarray(grad, mode='L')

plt.figure(figsize=(12, 2))
plt.imshow(gradient_image, cmap='gray', vmin=0, vmax=255)
plt.title('512x64 grayscale gradient')
plt.axis('off')
plt.show()

## Bayer Ordered Dithering

Ordered dithering compares the image against a repeating threshold matrix. Bayer matrices distribute thresholds in a balanced recursive pattern, giving a very regular texture whose visibility depends on matrix size.

In [ ]:
sizes = [2, 4, 8]
matrices = [bayer_matrix(size) for size in sizes]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, size, matrix in zip(axes, sizes, matrices):
    im = ax.imshow(matrix, cmap='viridis')
    ax.set_title(f'{size}x{size} Bayer matrix')
    for y in range(matrix.shape[0]):
        for x in range(matrix.shape[1]):
            ax.text(x, y, int(matrix[y, x]), ha='center', va='center', color='white', fontsize=8)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7)
plt.tight_layout()
plt.show()

In [ ]:
bayer2 = ordered_dither(gradient_image, matrix_size=2)
bayer4 = ordered_dither(gradient_image, matrix_size=4)
bayer8 = ordered_dither(gradient_image, matrix_size=8)

fig, axes = plt.subplots(3, 1, figsize=(12, 4))
for ax, title, result in zip(axes, ['Bayer 2x2', 'Bayer 4x4', 'Bayer 8x8'], [bayer2, bayer4, bayer8]):
    ax.imshow(result, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Floyd-Steinberg Error Diffusion

Floyd-Steinberg diffuses quantization error forward to neighboring pixels. The kernel is compact and efficient, and it often gives a sharp, classic newspaper-style binary image with distinctive diagonal texture.

In [ ]:
fs_kernel = np.array([
    [0, 0, 0],
    [0, 0, 7],
    [3, 5, 1],
], dtype=np.float32) / 16.0
fs_result = floyd_steinberg(gradient_image)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(fs_kernel, cmap='magma')
axes[0].set_title('Floyd-Steinberg kernel')
for y in range(fs_kernel.shape[0]):
    for x in range(fs_kernel.shape[1]):
        axes[0].text(x, y, f'{fs_kernel[y, x]:.2f}', ha='center', va='center', color='white')
axes[1].imshow(fs_result, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Floyd-Steinberg result')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Atkinson Dithering - the Mac algorithm

Atkinson dithering spreads less total error than Floyd-Steinberg, so it tends to keep highlights a bit lighter and produce a softer, more airy texture. It became famous on early Macintosh displays and printers.

In [ ]:
atkinson_result = atkinson(gradient_image)

fig, axes = plt.subplots(2, 1, figsize=(12, 4))
axes[0].imshow(fs_result, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Floyd-Steinberg')
axes[0].axis('off')
axes[1].imshow(atkinson_result, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Atkinson')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Jarvis-Judice-Ninke and Sierra

These larger kernels spread error more widely than Floyd-Steinberg. The wider diffusion reduces some coarse artifacts, but it also costs more computation and can soften local contrast.

In [ ]:
jjn_result = jjn(gradient_image)
sierra_result = sierra(gradient_image)

fig, axes = plt.subplots(2, 1, figsize=(12, 4))
axes[0].imshow(jjn_result, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Jarvis-Judice-Ninke')
axes[0].axis('off')
axes[1].imshow(sierra_result, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Sierra')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Blue Noise Dithering

A blue-noise style threshold pattern avoids strong low-frequency repetition, so it often looks less obviously tiled than ordered dithering. Here we use a compact interleaved gradient noise pattern as a practical stand-in.

In [ ]:
blue_noise = ign_blue_noise(64, 512)
blue_result = blue_noise_dither(gradient_image)

fig, axes = plt.subplots(2, 1, figsize=(12, 4))
axes[0].imshow(blue_noise, cmap='viridis')
axes[0].set_title('IGN blue-noise-like threshold pattern')
axes[0].axis('off')
axes[1].imshow(blue_result, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Blue-noise dithered gradient')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## All Algorithms Side by Side

Putting every method on the same test image reveals differences in regularity, grain, edge sharpness, and mid-tone texture. This is especially useful when choosing a look for a stylized halftone or mosaic pipeline.

In [ ]:
comparison = {
    'Bayer 2x2': bayer2,
    'Bayer 4x4': bayer4,
    'Bayer 8x8': bayer8,
    'Floyd-Steinberg': fs_result,
    'Atkinson': atkinson_result,
    'JJN': jjn_result,
    'Sierra': sierra_result,
    'Blue noise': blue_result,
}

fig, axes = plt.subplots(4, 2, figsize=(14, 10))
for ax, (name, result) in zip(axes.ravel(), comparison.items()):
    ax.imshow(result, cmap='gray', vmin=0, vmax=1)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Multi-level Quantization

Dithering is not limited to binary output. By quantizing to 4, 8, or 16 gray levels before diffusing the error, we can trade a richer tonal scale for more visibly stepped microstructure.

In [ ]:
fs_2 = floyd_steinberg(gradient_image, levels=2)
fs_4 = floyd_steinberg(gradient_image, levels=4)
fs_8 = floyd_steinberg(gradient_image, levels=8)
fs_16 = floyd_steinberg(gradient_image, levels=16)

fig, axes = plt.subplots(4, 1, figsize=(12, 5))
for ax, title, result in zip(axes, ['2 levels', '4 levels', '8 levels', '16 levels'], [fs_2, fs_4, fs_8, fs_16]):
    ax.imshow(result, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Floyd-Steinberg with {title}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Zoom In: Pattern Structure

A cropped view makes the local pattern obvious. Ordered methods reveal repeating cells, while error-diffusion methods show directional textures and worm-like structures that are hard to notice at full scale.

In [ ]:
x0, x1 = 224, 288
zoomed = {name: result[:, x0:x1] for name, result in comparison.items()}

fig, axes = plt.subplots(4, 2, figsize=(10, 10))
for ax, (name, result) in zip(axes.ravel(), zoomed.items()):
    ax.imshow(result, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Analysis - Discuss the visual properties, tradeoffs of each algorithm

- **Small Bayer matrices** are fast and predictable, but their periodic structure is easy to see.
- **Larger Bayer matrices** reduce coarse repetition at the cost of a larger visible tile.
- **Floyd-Steinberg** is compact, high contrast, and classic, though it can create directional artifacts.
- **Atkinson** feels lighter and more open because it diffuses less total error.
- **JJN and Sierra** spread error more broadly, often smoothing coarse artifacts but increasing computation.
- **Blue-noise thresholding** looks less grid-like than Bayer and less directional than many diffusion kernels.
- **Multi-level diffusion** is useful when the output device supports more than pure black and white.